# Iris Unsupervised Clustering

This notebook explores Iris using unsupervised methods only.
The target column is held out from all fitting steps and used
only for post-hoc evaluation metrics.

In [3]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd()
if not (project_root / 'src').exists() and project_root.parent.exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.analysis.kmeans import fit_kmeans
from src.analysis.kmeans import posthoc_label_metrics
from src.analysis.kmeans import score_kmeans
from src.utils.data import load_iris_features

sns.set_theme(style='whitegrid')

In [4]:
X, y = load_iris_features()
print('Feature shape:', X.shape)
print('Target shape:', y.shape)
X.head()

Feature shape: (150, 4)
Target shape: (150,)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [ ]:
missing = X.isna().sum()
print('Missing values by column:')
print(missing)

X.describe().T

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaled feature matrix shape:', X_scaled.shape)

In [ ]:
plt.figure(figsize=(6, 5))
corr = X.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap='viridis', square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(X, corner=True, diag_kind='hist')
plt.suptitle('Pairplot (no target hue)', y=1.02)
plt.show()

In [ ]:
k_values = list(range(2, 9))
inertias = []
silhouettes = []

for k in k_values:
    model = fit_kmeans(X_scaled, n_clusters=k, random_state=42)
    inertias.append(float(model.inertia_))
    score = score_kmeans(X_scaled, model.labels_)
    silhouettes.append(score['silhouette'])

k_best = k_values[int(np.argmax(silhouettes))]
print('Best k by silhouette:', k_best)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_values, inertias, marker='o')
axes[0].set_title('Elbow Curve')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')

axes[1].plot(k_values, silhouettes, marker='o', color='teal')
axes[1].set_title('Silhouette by k')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette')

plt.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.8, s=40)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA Projection (Unlabeled)')
plt.tight_layout()
plt.show()

In [ ]:
final_model = fit_kmeans(X_scaled, n_clusters=k_best, random_state=42)
cluster_labels = final_model.labels_
unsupervised_metrics = score_kmeans(X_scaled, cluster_labels)
pd.Series(unsupervised_metrics, name='value')

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=cluster_labels,
    palette='tab10',
    legend='full',
)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA Projection Colored by Cluster Labels')
plt.tight_layout()
plt.show()

In [ ]:
posthoc = posthoc_label_metrics(
    y_true=y.to_numpy(),
    cluster_labels=cluster_labels,
)

summary = {
    'adjusted_rand_index': posthoc['adjusted_rand_index'],
    'normalized_mutual_info': posthoc['normalized_mutual_info'],
    'v_measure': posthoc['v_measure'],
    'mapped_accuracy': posthoc['mapped_accuracy'],
    'mapped_macro_precision': posthoc['mapped_macro_precision'],
    'mapped_macro_recall': posthoc['mapped_macro_recall'],
    'mapped_macro_f1': posthoc['mapped_macro_f1'],
}

print('Post-hoc metrics (evaluation only):')
for key, value in summary.items():
    print(f'- {key}: {value:.4f}')

report_df = pd.DataFrame(posthoc['mapped_classification_report']).T
report_df

## Recap

- Model fitting, scaling, PCA, and k selection used only feature data.
- Unsupervised metrics evaluate compactness and separation without labels.
- Target-based metrics are post-hoc diagnostics only and are not used
  for training decisions.